<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_7_SEQ_COV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from copy import deepcopy

# classic 14-instance PlayTennis concept learning dataset
records = [
    {"Outlook": "Sunny",    "Temperature": "Hot",  "Humidity": "High",   "Wind": "Weak",   "PlayTennis": "No"},
    {"Outlook": "Sunny",    "Temperature": "Hot",  "Humidity": "High",   "Wind": "Strong", "PlayTennis": "No"},
    {"Outlook": "Overcast", "Temperature": "Hot",  "Humidity": "High",   "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Rain",     "Temperature": "Mild", "Humidity": "High",   "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Rain",     "Temperature": "Cool", "Humidity": "Normal", "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Rain",     "Temperature": "Cool", "Humidity": "Normal", "Wind": "Strong", "PlayTennis": "No"},
    {"Outlook": "Overcast", "Temperature": "Cool", "Humidity": "Normal", "Wind": "Strong", "PlayTennis": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Mild", "Humidity": "High",   "Wind": "Weak",   "PlayTennis": "No"},
    {"Outlook": "Sunny",    "Temperature": "Cool", "Humidity": "Normal", "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Rain",     "Temperature": "Mild", "Humidity": "Normal", "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Sunny",    "Temperature": "Mild", "Humidity": "Normal", "Wind": "Strong", "PlayTennis": "Yes"},
    {"Outlook": "Overcast", "Temperature": "Mild", "Humidity": "High",   "Wind": "Strong", "PlayTennis": "Yes"},
    {"Outlook": "Overcast", "Temperature": "Hot",  "Humidity": "Normal", "Wind": "Weak",   "PlayTennis": "Yes"},
    {"Outlook": "Rain",     "Temperature": "Mild", "Humidity": "High",   "Wind": "Strong", "PlayTennis": "No"},
]

dataset = pd.DataFrame(records)
TARGET_COL = "PlayTennis"
POS_CLASS = "Yes"
FEATURE_COLS = [c for c in dataset.columns if c != TARGET_COL]

print("PlayTennis dataset:")
print(dataset.to_string(index=False))

In [ ]:
# helper functions to evaluate and format rule conditions
def rule_matches(rule, row):
    # returns True if example satisfies every attribute=value condition in rule
    return all(row[attr] == val for attr, val in rule.items())

def score_rule(rule, examples):
    # calculates positive coverage, negative coverage, and rule precision
    covered = examples[examples.apply(lambda r: rule_matches(rule, r), axis=1)]
    pos_covered = int((covered[TARGET_COL] == POS_CLASS).sum())
    neg_covered = int((covered[TARGET_COL] != POS_CLASS).sum())
    total = pos_covered + neg_covered
    prec = pos_covered / total if total > 0 else 0.0
    return pos_covered, neg_covered, prec

def format_rule(rule):
    if not rule:
        return "IF <no conditions> THEN PlayTennis=Yes"
    conditions = " AND ".join(f"{attr}={val}" for attr, val in rule.items())
    return f"IF {conditions} THEN PlayTennis=Yes"

# Sequential Covering: finds best rule via general-to-specific beam search
# Beam search retains top 'beam_width' candidates to prevent getting trapped in local optima
def find_best_rule(examples, beam_width=4, max_conditions=None):
    beam = [{}] # start with the most general rule (empty condition set)
    best_rule = {}
    best_precision = 0.0

    while True:
        candidates = []
        for candidate in beam:
            # specialize candidate by adding one new attribute constraint
            for feat in FEATURE_COLS:
                if feat in candidate:
                    continue
                if max_conditions and len(candidate) >= max_conditions:
                    continue
                for val in examples[feat].unique():
                    new_rule = {**candidate, feat: val}
                    pos, neg, prec = score_rule(new_rule, examples)
                    if pos > 0:
                        candidates.append((new_rule, prec, pos))

        if not candidates:
            break

        # sort candidates by precision first, then by positive coverage
        candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
        beam = [c[0] for c in candidates[:beam_width]]
        top_rule, top_prec, top_pos = candidates[0]

        if top_prec > best_precision:
            best_rule = top_rule
            best_precision = top_prec
        else:
            break

    return best_rule

def run_covering(data, min_precision=0.9, beam_width=4):
    remaining = data.copy()
    rules = []
    cov_log = []

    # loop until all positive instances are covered
    while (remaining[TARGET_COL] == POS_CLASS).any():
        rule = find_best_rule(remaining, beam_width=beam_width)
        pos, neg, prec = score_rule(rule, remaining)

        if prec < min_precision or pos == 0:
            break

        rules.append(rule)
        cov_log.append({"rule": format_rule(rule), "pos": pos, "neg": neg, "prec": prec})
        print(f"Learned: {format_rule(rule)}  [prec={prec:.2f}, pos={pos}, neg={neg}]")

        # drop positive instances covered by this rule so next iteration focuses on remaining ones
        covered_mask = remaining.apply(lambda r: rule_matches(rule, r), axis=1)
        remaining = remaining[~(covered_mask & (remaining[TARGET_COL] == POS_CLASS))].reset_index(drop=True)

    return rules, cov_log

learned_rules, cov_log = run_covering(dataset, min_precision=0.9, beam_width=4)

print("
Final Rule Set (Sequential Covering):")
for i, r in enumerate(learned_rules, 1):
    pos, neg, prec = score_rule(r, dataset)
    print(f"  Rule {i}: {format_rule(r)} [Precision={prec:.2f}, Pos={pos}, Neg={neg}]")

# plot coverage and precision per learned rule
if cov_log:
    labels = [f"Rule {i+1}" for i in range(len(cov_log))]
    pos_vals = [c["pos"] for c in cov_log]
    neg_vals = [c["neg"] for c in cov_log]
    prec_vals = [c["prec"] for c in cov_log]

    x = range(len(labels))
    w = 0.35
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar([i - w/2 for i in x], pos_vals, w, color="#4CAF50", alpha=0.85, label="Positives Covered")
    ax1.bar([i + w/2 for i in x], neg_vals, w, color="#F44336", alpha=0.85, label="Negatives Covered")
    ax1.set_xlabel("Rule")
    ax1.set_ylabel("Count")
    ax1.set_title("Sequential Covering — Coverage per Rule")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels, rotation=15)
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(list(x), prec_vals, marker="D", color="navy", linewidth=2, label="Precision")
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel("Precision")
    ax2.axhline(0.9, color="gray", linestyle="--", alpha=0.6, label="Min Precision (0.9)")
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
# AQ Algorithm: seed-based rule induction
# For an uncovered positive seed, generates a STAR (set of all maximally general rules
# consistent with the seed but excluding negative instances)

def aq_rule_quality(rule, examples):
    # score favors high precision and penalizes unnecessary rule complexity (length)
    pos, neg, prec = score_rule(rule, examples)
    return prec - 0.1 * len(rule)

def generate_star(seed_example, negatives):
    star = [{}]  # start with the empty/general rule
    for _, neg_row in negatives.iterrows():
        new_star = []
        for rule in star:
            if not rule_matches(rule, neg_row):
                new_star.append(rule)
                continue
            # specialize rule to separate seed from this negative example
            for feat in FEATURE_COLS:
                if feat in rule:
                    continue
                seed_val = seed_example[feat]
                neg_val = neg_row[feat]
                if seed_val != neg_val:
                    specialised = {**rule, feat: seed_val}
                    if rule_matches(specialised, seed_example):
                        new_star.append(specialised)
        # remove duplicate candidates
        seen, deduped = set(), []
        for r in new_star:
            key = tuple(sorted(r.items()))
            if key not in seen:
                seen.add(key)
                deduped.append(r)
        star = deduped
    return star

def aq_algorithm(data, min_precision=0.9):
    remaining = data.copy()
    aq_rules = []
    aq_log = []
    iteration = 0

    while (remaining[TARGET_COL] == POS_CLASS).any():
        iteration += 1
        positives = remaining[remaining[TARGET_COL] == POS_CLASS]
        negatives = remaining[remaining[TARGET_COL] != POS_CLASS]

        # pick first uncovered positive instance as seed
        seed = positives.iloc[0]
        star = generate_star(seed, negatives)

        if not star:
            print(f"[AQ Iter {iteration}] No consistent rules found. Stopping.")
            break

        # choose highest quality rule from the star
        best_rule = max(star, key=lambda r: aq_rule_quality(r, remaining))
        pos, neg, prec = score_rule(best_rule, remaining)

        print(f"[AQ Iter {iteration}] {format_rule(best_rule)}")
        print(f"             Precision={prec:.2f}  Pos={pos}  Neg={neg}")

        aq_rules.append(best_rule)
        aq_log.append({"rule": format_rule(best_rule), "pos": pos, "neg": neg, "prec": prec})

        # remove covered positives
        covered_mask = remaining.apply(lambda r: rule_matches(best_rule, r), axis=1)
        remaining = remaining[~(covered_mask & (remaining[TARGET_COL] == POS_CLASS))].reset_index(drop=True)

    return aq_rules, aq_log

aq_rules, aq_log = aq_algorithm(dataset, min_precision=0.9)

print("
Final AQ Rule Set:")
for idx, r in enumerate(aq_rules, 1):
    pos, neg, prec = score_rule(r, dataset)
    print(f"  Rule {idx}: {format_rule(r)} [Precision={prec:.2f}, Pos={pos}, Neg={neg}]")

In [ ]:
# visual summary of AQ rules: coverage bar chart, precision bar, and summary table
if aq_log:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    aq_labels = [f"AQ Rule {i+1}" for i in range(len(aq_log))]
    aq_x = range(len(aq_labels))
    aq_pos = [e["pos"] for e in aq_log]
    aq_neg = [e["neg"] for e in aq_log]
    aq_prec = [e["prec"] for e in aq_log]

    w = 0.35
    axes[0].bar([i - w/2 for i in aq_x], aq_pos, w, color="#2196F3", alpha=0.85, label="Positives")
    axes[0].bar([i + w/2 for i in aq_x], aq_neg, w, color="#FF9800", alpha=0.85, label="Negatives")
    axes[0].set_title("AQ Algorithm — Coverage per Rule")
    axes[0].set_xlabel("Rule")
    axes[0].set_ylabel("Count")
    axes[0].set_xticks(list(aq_x))
    axes[0].set_xticklabels(aq_labels, rotation=20, ha="right")
    axes[0].legend()

    # precision horizontal bar chart
    axes[1].barh(aq_labels[::-1], aq_prec[::-1], color="#9C27B0", alpha=0.85)
    axes[1].set_xlim(0, 1.05)
    axes[1].axvline(0.9, color="red", linestyle="--", label="Min Precision (0.9)")
    axes[1].set_title("AQ Algorithm — Precision per Rule")
    axes[1].set_xlabel("Precision")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # summary table
    fig2, ax_tbl = plt.subplots(figsize=(12, max(2, len(aq_log) * 0.6 + 1)))
    ax_tbl.axis("off")
    tbl_data = [[e["rule"], e["pos"], e["neg"], f"{e['prec']:.2f}"] for e in aq_log]
    col_hdrs = ["Rule", "Pos Covered", "Neg Covered", "Precision"]
    tbl = ax_tbl.table(cellText=tbl_data, colLabels=col_hdrs, cellLoc="left", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width([0, 1, 2, 3])
    ax_tbl.set_title("AQ Rule Summary", pad=12)
    plt.tight_layout()
    plt.show()